# Bài học 20: Web cơ bản cho người không phải developer

**Thời gian:** ~30 phút | **Chi phí:** $0 (không gọi API)

Trong bài tiếp theo, bạn sẽ thấy giao diện web của sản phẩm — một frontend React giao tiếp với backend FastAPI. Bài học này dạy các khái niệm cần thiết để hiểu chuyện gì đang xảy ra.

Bạn không cần trở thành web developer. Bạn cần hiểu **kiến trúc** để có thể chỉ đạo Claude Code thực hiện thay đổi.

## Web app là gì?

Một web app gồm hai phần:
- **Frontend** (phía người dùng) — những gì bạn thấy trên trình duyệt. HTML, CSS, JavaScript. Trong dự án này: React.
- **Backend** (phía máy chủ) — bộ máy xử lý yêu cầu. Code Python. Trong dự án này: FastAPI + các agent.

```
[Trình duyệt]  ↔  [Backend Server]
   Frontend           Python + Agents
   (React)            (FastAPI)
```

Ví dụ: Một nhà hàng.
- **Frontend** = phòng ăn (menu, bàn ghế, nhân viên phục vụ)
- **Backend** = nhà bếp (đầu bếp, nguyên liệu, công thức)
- Nhân viên phục vụ mang **yêu cầu** (request) vào bếp và mang **kết quả** (response) ra

Khi bạn gõ tin nhắn trong giao diện chat rồi bấm gửi, bạn là khách hàng đang đặt món. Tin nhắn được gửi tới backend (nhà bếp), team agent xử lý nó (đầu bếp nấu), và kết quả quay lại màn hình (nhân viên mang đồ ăn ra bàn).

## API là gì?

**API** (Application Programming Interface — giao diện lập trình ứng dụng) — một tập hợp các **endpoint** (URL) mà backend cung cấp cho frontend gọi tới.

Ví dụ từ dự án:

| Endpoint | Method | Chức năng |
|----------|--------|----------|
| `/api/articles` | GET | Liệt kê tất cả bài viết |
| `/api/articles/{id}` | POST | Cập nhật một bài viết |
| `/teams/seo-workspace/runs` | POST | Gửi tin nhắn chat tới team |
| `/health` | GET | Kiểm tra server có đang chạy không |

**GET** = "cho tôi dữ liệu" (giống đọc menu)

**POST** = "đây là dữ liệu, hãy xử lý" (giống đặt món)

Mỗi lời gọi API gồm một **request** (từ frontend) và một **response** (từ backend):

```
Frontend: "GET /api/articles"
Backend:  [đọc content/articles.json, trả về danh sách]
Frontend: hiển thị danh sách bài viết ở sidebar
```

Bạn đã hiểu pattern này rồi — đó chính là cách DataForSEO tool hoạt động! Khi Content Writer tìm kiếm trên web, nó gọi API tới các endpoint của DataForSEO. Cùng khái niệm, khác server.

In [ ]:
# Lời gọi API chỉ là các request có cấu trúc
# Đây là những gì xảy ra khi frontend yêu cầu danh sách bài viết

# Frontend gửi request này:
request = {
    "method": "GET",
    "url": "/api/articles",
    "headers": {"Content-Type": "application/json"},
}

# Backend xử lý và trả về response này:
response = {
    "status": 200,  # 200 = OK
    "body": [
        {"id": "on-page-seo-meta-tags", "topic": "On-Page SEO Meta Tags", "status": "review"},
        {"id": "internal-linking", "topic": "Internal Linking Strategy", "status": "review"},
    ]
}

print(f"Request: {request['method']} {request['url']}")
print(f"Response status: {response['status']}")
print(f"Articles returned: {len(response['body'])}")
for article in response['body']:
    print(f"  - {article['topic']} ({article['status']})")

## JSON — Ngôn ngữ của API

Frontend và backend giao tiếp bằng **JSON** — cùng định dạng mà bạn đã dùng với `json.loads()` và `json.dumps()` từ Bài 12.

```json
{
    "topic": "SEO Tips",
    "word_count": 1500,
    "status": "review",
    "keywords": ["seo", "tips", "beginners"]
}
```

Đây là lý do tại sao tất cả các hàm storage đều trả về chuỗi JSON — chúng được thiết kế để hoạt động với cả agent LẪN web API.

## Frontend và backend kết nối như thế nào

Dự án chạy hai server:
- **Backend**: `python output/backend/serve.py` → chạy trên **port 7777**
- **Frontend**: `cd output/frontend && npm run dev` → chạy trên **port 5173**

Frontend (port 5173) cần giao tiếp với backend (port 7777). Vite dev server (công cụ build của frontend) **proxy** (chuyển tiếp) các request API:

```
Browser (port 5173) → Vite proxy → Backend (port 7777)
                        ↓
              Chuyển tiếp /api/*, /teams/*, /health
```

Nghĩa là:
- Bạn mở `http://localhost:5173` trên trình duyệt
- Khi React gửi request tới `/api/articles`, Vite chuyển tiếp tới `http://localhost:7777/api/articles`
- Response quay lại qua cùng đường đi

**Bạn không cần cấu hình gì** — tất cả đã được thiết lập sẵn trong `output/frontend/vite.config.js`. Nhưng hiểu cơ chế này giúp khi debug ("tại sao frontend không kết nối được backend?").

## SSE — Cách streaming hoạt động

Khi bạn chat với ChatGPT, chữ xuất hiện từng từ một. Đó là **SSE** (Server-Sent Events — sự kiện được gửi từ server) — cách server **stream** (truyền) dữ liệu tới trình duyệt theo thời gian thực.

Giao diện chat của dự án dùng cùng kỹ thuật:
1. Frontend gửi tin nhắn qua POST
2. Backend khởi động team agent
3. Khi agent tạo văn bản, backend gửi **từng đoạn nhỏ** (chunk) qua SSE
4. Frontend hiển thị mỗi chunk khi nó tới → bạn thấy chữ xuất hiện theo thời gian thực

Không có SSE: bạn phải đợi 30–90 giây nhìn vòng xoay loading, rồi nhận toàn bộ kết quả một lúc.

Có SSE: chữ bắt đầu xuất hiện trong vài giây, streaming khi agent viết.

```
Frontend: "Write an article about SEO tips" (POST)
Backend:  SSE chunk: "# SEO Tips..."
Backend:  SSE chunk: "\n\nSearch engine optimization..."
Backend:  SSE chunk: " is essential for..."
          ... (tiếp tục streaming)
Backend:  SSE chunk: [DONE]
```

## Tổng kết bài 20

| Khái niệm | Ý nghĩa | Trong dự án |
|-----------|---------|-------------|
| **Frontend** | Giao diện phía trình duyệt (React) | `output/frontend/` |
| **Backend** | Logic phía server (Python) | `output/backend/serve.py` |
| **API** | Các endpoint mà frontend gọi tới | `/api/articles`, `/teams/...` |
| **GET / POST** | Đọc dữ liệu / Gửi dữ liệu | Liệt kê bài viết / Gửi tin nhắn chat |
| **JSON** | Định dạng dữ liệu cho giao tiếp | Cùng định dạng từ storage tools |
| **Proxy** | Vite chuyển tiếp request tới backend | Port 5173 → port 7777 |
| **SSE** | Streaming theo thời gian thực | Chữ chat xuất hiện từng từ |

**Những gì bạn KHÔNG cần biết**: Cách React render component, cách FastAPI xử lý routing nội bộ, cách SSE được implement bằng JavaScript.

**Những gì bạn CẦN biết**: Kiến trúc — cái gì giao tiếp với cái gì, và dữ liệu chảy đi đâu. Vậy là đủ để chỉ đạo Claude Code.

**Bài tiếp theo:** Giao diện web — xem tất cả những điều này hoạt động thực tế với sản phẩm.

## Bài tập

Nối mỗi thành phần với vai trò của nó:

In [ ]:
# Bài tập: Nối mỗi thành phần với vai trò của nó

components = {
    "React": "___",           # frontend hay backend?
    "FastAPI": "___",         # frontend hay backend?
    "serve.py": "___",        # nó làm gì?
    "/api/articles": "___",   # đây là loại gì?
    "port 7777": "___",       # cái gì chạy ở đây?
    "port 5173": "___",       # cái gì chạy ở đây?
    "SSE": "___",             # nó cho phép gì?
    "JSON": "___",            # nó được dùng để làm gì?
}

for component, role in components.items():
    print(f"  {component}: {role}")

<details>
<summary>Bấm để xem đáp án</summary>

- **React**: frontend (giao diện trình duyệt)
- **FastAPI**: backend (framework web Python)
- **serve.py**: điểm vào backend (khởi động web server với các agent)
- **/api/articles**: API endpoint (URL mà frontend gọi để lấy dữ liệu bài viết)
- **port 7777**: backend server
- **port 5173**: frontend dev server
- **SSE**: streaming theo thời gian thực (chữ xuất hiện từng từ)
- **JSON**: định dạng dữ liệu cho giao tiếp API

</details>